In [12]:
T = CartanType(['A', 2])
W = WeylGroup(T,prefix="s")
[s1, s2] = W.simple_reflections()

WCR = WeylCharacterRing(T, style="coroots", base_ring=QQ)
R = WeightRing(WCR)
# EXPI MODIFIED, THIS CODE ONLY WORKS IN TYPE A
f = [w.coerce_to_sl() for w in WCR.fundamental_weights()]

n = len(W.simple_reflections())
e = s1^2
w0 = W.long_element()
S.<q> = LaurentPolynomialRing(QQ)
KL = KazhdanLusztigPolynomial(W,q)
q = 29

def Iw(w):
    a = []
    for i in range(1, n+1):
        if w.has_right_descent(i):
            a.append(i)
    return a

def Iwc(w):
    a = []
    for i in range(1, n+1):
        if not w.has_right_descent(i):
            #a.append(W.simple_reflections()[i])
            a.append(i)
    return a

def expI(l):
    a = R.one()
    for i in l:
        a = a * R(f[i-1])
    return a

def esub(w):
    return expI(Iw(w)).weyl_group_action(w)

def eqsub(w):
    return expI(Iw(w)).weyl_group_action(w).scale(q)

def esup(w):
    return (-1)^(w.length())*expI(Iwc(w)).weyl_group_action(w)

def Iweight(w):
    wgt = 0
    for a in Iw(w):
        wgt += f[a-1]
    return wgt

rho = Iweight(w0)

def weightpairing(l):
    a = WCR.dot_reduce(l)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    b = b.scale(-1)
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

def woke_weightpairing(l):
    a = WCR.dot_reduce(l - rho)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def inprod(a, b):
    da = dict(a)
    db = dict(b)
    t = 0
    for v in da:
        for u in db:
            t += da[v] * db[u] * v.inner_product(u)
    return t

def rootprod(a, b):
    return 2*inprod(a,b)/inprod(b,b)
    
def dom(a):
    for b in R.simple_roots():
        if rootprod(a, R(b)) < 0:
            return False
    return True

def woke_pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*woke_weightpairing(l)
    return r

def pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    a = a.scale(-1)
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

def norma(a):
    k = list(dict(a).keys())
    return R(k[0])

d_sub = {}
d_sup = {}
d_qsub = {}
for i in range(len(W)):
    d_sub[i] = esub(W[i])
    d_sup[i] = esup(W[i])
    d_qsub[i] = eqsub(W[i])

SignalError: Segmentation fault

In [16]:
def build_matrix():
    m = []
    for i in range(len(W)):
        print(i + 1, "out of", len(W), end='\r')
        r = []
        for j in range(len(W)):
            r.append(woke_pairing(d_sub[i], d_sup[j]))
        m.append(r)
    return m
bigmat = build_matrix()
#bigmat[i][j] = woke_pairing(d_sub[i], d_sup[j]) for all i, j
alls = list(range(len(W)))
pops = list(range(len(W)))
order = []
while len(pops) > 0:
    for a in pops:
        good = True
        for b in pops:
            if b != a and bigmat[b][a] != 0:
                good = False
        if good:
            order.append(a)
            pops.remove(a)

order.reverse()
print("Computed the matrix of pairings and the upper triangular order.")

Computed the matrix of pairings and the upper triangular order.


In [17]:
def lc(i):
    a = bigmat[i][i]
    d = dict(WCR(a))
    if len(d) != 1:
        print("LC ERROR")
    else:
        return list(d.values())[0]

counter = 0
d_dual = {}
for j in order:
    counter += 1
    print(counter, "out of", len(W), end='\r')
    d_dual[j] = d_sup[j]/lc(j)
    for i in order:
        if  i != j and woke_pairing(d_sub[i], d_dual[j]) != 0:
            d_dual[j] = d_dual[j] - woke_pairing(d_sub[i], d_dual[j])*d_sup[i]/lc(i)
print("Computed the dual basis as the dictionary d_dual.")

Computed the dual basis as the dictionary d_dual.


In [18]:
#OPTIONAL CELL: CHECKING THE DUAL BASIS IS VALID.
valid = True
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    for j in range(len(W)):
        a = woke_pairing(d_sub[i], d_dual[j])
        if a != 0:
            if i != j or a != 1:
                print("ISSUE AT", i, j)
                valid = False
if valid:
    print("Checked dual basis, all is good.")
else:
    print("ISSUES! Dual basis is NOT correct.")

Checked dual basis, all is good.


In [19]:
#takes in a w and y
#tells you some information...
def churn(w, y):
    gottem = False
    for z in W:
        cand = (esub(w)*esup(y)).weyl_group_action(z)
        shift = cand * R(rho).scale(-1)
        if dom(norma(shift)):
            gottem = True
            print(w, y)
            print("zw =", z*w)
            print("zy =", z*y)
            print("y^(-1)w =", y.inverse()*w)
            print("z =", z)
            #print(esub(w).weyl_group_action(z))
            #print(esup(y).weyl_group_action(z))
            if dom(esub(w).weyl_group_action(z)) and dom(norma(esup(y).weyl_group_action(z))):
                print("Special! z*esub(w) and z*esup(y) both dominant,", esub(w).weyl_group_action(z), esup(y).weyl_group_action(z))
            print(cand)
            print(KL.P(w, y))
            print()
    if not gottem:
        print(w, y)
        print("Pairing is zero.")
        print(KL.P(w, y))
        print()

for w in W:
    for y in W:
        if w != y:
            churn(w, y)

1 s1*s2
Pairing is zero.
1

1 s2*s1
Pairing is zero.
1

1 s1
Pairing is zero.
1

1 s2
Pairing is zero.
1

1 s1*s2*s1
Pairing is zero.
1

s1*s2 1
Pairing is zero.
0

s1*s2 s2*s1
Pairing is zero.
0

s1*s2 s1
Pairing is zero.
0

s1*s2 s2
Pairing is zero.
0

s1*s2 s1*s2*s1
Pairing is zero.
1

s2*s1 1
Pairing is zero.
0

s2*s1 s1*s2
Pairing is zero.
0

s2*s1 s1
Pairing is zero.
0

s2*s1 s2
Pairing is zero.
0

s2*s1 s1*s2*s1
Pairing is zero.
1

s1 1
Pairing is zero.
0

s1 s1*s2
Pairing is zero.
1

s1 s2*s1
Pairing is zero.
1

s1 s2
Pairing is zero.
0

s1 s1*s2*s1
Pairing is zero.
1

s2 1
Pairing is zero.
0

s2 s1*s2
Pairing is zero.
1

s2 s2*s1
Pairing is zero.
1

s2 s1
Pairing is zero.
0

s2 s1*s2*s1
Pairing is zero.
1

s1*s2*s1 1
Pairing is zero.
0

s1*s2*s1 s1*s2
Pairing is zero.
0

s1*s2*s1 s2*s1
Pairing is zero.
0

s1*s2*s1 s1
Pairing is zero.
0

s1*s2*s1 s2
Pairing is zero.
0



1 + q


In [ ]:
newmat = []
lis = []
for i in range(len(W)):
    row = []
    for j in range(len(W)):
        a = bigmat[order[i]][order[j]]
        row.append(a)
        z = W[order[j]].inverse()*W[order[i]]
        l = [s3*s1*s2, s2*s3*s1]
        if i != j and a != 0:
            print(W[order[i]], W[order[j]], Iw(W[order[i]]), Iw(W[order[j]]))
            print(W[order[j]].inverse() * W[order[i]])
            print()
            lis.append([W[order[i]], W[order[j]]])
    newmat.append(row)
print(lis)
print(matrix(newmat))

In [6]:
print(d_dual)

{31: -b3(-2,1,-1) - b3(-3,1,1) + b3(0,0,-1) + 2*b3(0,-1,1) + b3(0,-2,3) + b3(-1,1,-1) + b3(-1,0,1) + b3(-1,-1,3) - b3(-2,2,-1) + b3(1,0,-1) + 2*b3(1,-1,1) + b3(1,-2,3) + b3(0,1,-1) + b3(0,0,1) + b3(0,-1,3) - b3(-1,2,-1) - b3(1,1,-1) - b3(0,1,1), 21: b3(0,0,-1) + b3(-1,2,-3) + 2*b3(-1,1,-1) + b3(1,0,-1) + b3(0,2,-3) + 2*b3(0,1,-1) + b3(1,1,-1), 44: -3*b3(0,0,0) - b3(-1,0,0) - b3(-2,2,-2) - b3(-1,1,-2) + b3(1,0,0) + b3(2,0,-2) - b3(0,1,0) - b3(0,2,-2) + b3(1,2,-4) + 2*b3(1,-1,2) - b3(1,1,0) + b3(1,-1,0) + b3(1,1,-4) + b3(1,-2,2) - 2*b3(-1,1,0) - 2*b3(-1,2,-2) - 2*b3(0,1,-2) - b3(0,-1,2), 7: b3(0,0,-1) + b3(-1,1,-1) + b3(2,-1,-1) + b3(1,0,-1) + b3(1,-1,1) + 2*b3(0,1,-1) - b3(0,0,1) + b3(3,-2,1) + b3(2,0,-1), 34: -b3(0,-1,1) - 2*b3(-1,0,1) - b3(-1,-1,3) - b3(1,-1,1) - 2*b3(0,0,1) - b3(0,-1,3) - b3(1,0,1), 29: -2*b3(0,0,0) + b3(1,0,0) - b3(0,1,0) + b3(1,-1,2) - b3(-1,1,0) - b3(-1,2,-2) - b3(-1,0,2) - b3(-2,2,0) - b3(-2,1,2) - b3(0,1,-2) - b3(0,-1,2), 13: b3(0,-1,1) + b3(-1,0,1) + b3(1,-1,1)

In [21]:
for w in W:
    print(w, esub(w))

1 a2(0,0)
s1*s2 a2(-1,0)
s2*s1 a2(0,-1)
s1 a2(-1,1)
s2 a2(1,-1)
s1*s2*s1 a2(-1,-1)


In [45]:
p = R(-1,0) + R(0,1) + R(1,-1)
a = R(1,0) + R(-1,1)
b = R(1,-1) + R(0,1)
c = R(0,0)
print(woke_pairing(p,c))

0


In [34]:
print(WCR.demazure_character(2*f[0] + 1*f[1], word=[1,2]))
#HYPE this is it! ok cool. 

a2(-3,2) + a2(-1,1) + a2(-2,3) + a2(1,0) + a2(0,2) + a2(3,-1) + a2(2,1)
